# Delivery 2: EMG movimiento vs. reposo con Machine Learning
Pasos que pide el curso (videos 9 y 10, *WHAT-YOU-HAND-IN*):
1. Cargar los datos (una fila = una repetición, ya con sus features).
2. Elegir **dos features**.
3. Separar **entrenamiento** y **prueba en una sesión nueva** (la prueba nunca entrena).
4. Calcular los **dos pisos**: azar (chance) y umbral (threshold).
5. Entrenar **k vecinos más cercanos (kNN)** y medir su accuracy en la sesión nueva.
6. **Matriz de confusión** y figura del espacio de features.

## Paso 1. Cargar el Excel
En Colab: sube `EMG_dataset_ML.xlsx` con el ícono de carpeta (izquierda) y corre la celda. Se usa la hoja `datos`: una fila = una fase de 5 s; `etiqueta` 0 = reposo, 1 = movimiento; `conjunto` dice qué filas entrenan y cuáles prueban.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

ARCHIVO = "EMG_dataset_ML.xlsx"
datos = pd.read_excel(ARCHIVO, sheet_name="datos")

print(datos.groupby(["sesion", "etiqueta"]).size().unstack())  # cuántas de cada clase por sesión
datos.head()

## Paso 2. Elegir dos features
Tienes rms_mV, mav_mV y env_mV (todas de **tiempo**). Mira la correlación: si dos features casi valen lo mismo (≈1.0), la segunda no le da información nueva al modelo.

In [ ]:
print(datos[["rms_mV", "mav_mV", "env_mV"]].corr().round(4))

FEATURES = ["rms_mV", "mav_mV"]   # las dos features del curso en tiempo (video 10, diapositiva 3)

## Paso 3. Separar entrenamiento y prueba por SESIÓN
La prueba justa es una **sesión nueva** (electrodos quitados y puestos de nuevo). Nunca mezcles filas de la misma sesión en entrenamiento y prueba: eso es fuga de datos.

In [ ]:
# La columna 'conjunto' del Excel decide: S1 y S2 = entrenamiento, S3 = prueba (sesión nueva)
entreno = datos[datos["conjunto"] == "entrenamiento"]
prueba  = datos[datos["conjunto"] == "prueba"]
SESION_PRUEBA = ", ".join(prueba["sesion"].unique())

X_entreno, y_entreno = entreno[FEATURES], entreno["etiqueta"]
X_prueba,  y_prueba  = prueba[FEATURES],  prueba["etiqueta"]
print("Entrenamiento:", len(entreno), "filas | Prueba:", len(prueba), "filas")

## Paso 4. Los dos pisos
- **Piso de azar:** adivinar. Con tantos movimientos como reposos es 50 %.
- **Piso de umbral:** una sola regla `si MAV > umbral → movimiento`, sin ML. El umbral se elige **solo con entrenamiento**.

In [ ]:
# Piso de azar: predecir siempre la clase más común del entrenamiento
clase_comun = y_entreno.mode()[0]
piso_azar = accuracy_score(y_prueba, np.full(len(y_prueba), clase_comun))

# Piso de umbral: probar cada valor de MAV del entrenamiento como umbral y quedarse con el mejor
mejor_umbral, mejor_acc = None, 0
for u in np.sort(entreno["mav_mV"].unique()):
    acc = accuracy_score(y_entreno, (entreno["mav_mV"] > u).astype(int))
    if acc > mejor_acc:
        mejor_umbral, mejor_acc = u, acc
piso_umbral = accuracy_score(y_prueba, (prueba["mav_mV"] > mejor_umbral).astype(int))

print(f"Piso de azar:   {piso_azar:.0%}")
print(f"Piso de umbral: {piso_umbral:.0%}  (MAV > {mejor_umbral:.2f} mV)")

## Paso 5. Entrenar kNN y probar en la sesión nueva
`StandardScaler` pone RMS y MAV en la misma escala para que una no domine la distancia. k = 3 vecinos (video 9).

In [ ]:
modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
modelo.fit(X_entreno, y_entreno)            # aprende SOLO con entrenamiento

y_pred = modelo.predict(X_prueba)           # predice la sesión que nunca vio
acc_modelo = accuracy_score(y_prueba, y_pred)

print(f"Piso de azar:   {piso_azar:.0%}")
print(f"Piso de umbral: {piso_umbral:.0%}")
print(f"Modelo kNN:     {acc_modelo:.0%}  (sesión nueva {SESION_PRUEBA})")

## Paso 6. Matriz de confusión y espacio de features

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

# Matriz de confusión: filas = clase real, columnas = lo que dijo el modelo
ConfusionMatrixDisplay(confusion_matrix(y_prueba, y_pred),
                       display_labels=["Reposo", "Movimiento"]).plot(ax=ax[0], colorbar=False)
ax[0].set_title(f"Matriz de confusión, sesión nueva ({acc_modelo:.0%})")

# Espacio de features: cada repetición es un punto
for etiqueta, nombre, color in [(0, "Reposo", "tab:blue"), (1, "Movimiento", "tab:orange")]:
    e = entreno[entreno["etiqueta"] == etiqueta]
    p = prueba[prueba["etiqueta"] == etiqueta]
    ax[1].scatter(e["mav_mV"], e["rms_mV"], c=color, alpha=0.4, label=f"{nombre} (entreno)")
    ax[1].scatter(p["mav_mV"], p["rms_mV"], c=color, marker="x", label=f"{nombre} (prueba)")
ax[1].axvline(mejor_umbral, ls="--", c="gray", label="Piso de umbral")
ax[1].set_xlabel("MAV (mV)"); ax[1].set_ylabel("RMS (mV)"); ax[1].legend(fontsize=8)
ax[1].set_title("Espacio de features")
plt.tight_layout(); plt.savefig("resultados_delivery2.png", dpi=150); plt.show()

## Extra (opcional). Repetir con cada sesión como prueba
Muestra que el resultado no depende de qué sesión elegiste.

In [ ]:
for s in datos["sesion"].unique():
    a, b = datos[datos["sesion"] != s], datos[datos["sesion"] == s]
    modelo.fit(a[FEATURES], a["etiqueta"])
    print(f"Prueba en sesión {s}: {accuracy_score(b['etiqueta'], modelo.predict(b[FEATURES])):.0%}")